In [1]:
!pip install -q streamlit plotly pandas numpy reportlab

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os

BASE_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Retail Sales Prediction"
)

APP_DIR = os.path.join(BASE_DIR, "streamlit_dashboard")
os.makedirs(APP_DIR, exist_ok=True)

print("Dashboard folder:", APP_DIR)

Mounted at /content/drive
Dashboard folder: /content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard


In [9]:

%%writefile /content/app.py

import os
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import plotly.express as px
import streamlit as st

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet

import sys
import plotly.graph_objects as go

APP_DIR = os.path.dirname(os.path.abspath(__file__))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

from lstm_forecast import forecast_revenue, LOOKBACK


st.set_page_config(
    page_title="RetailPulse Analytics",
    page_icon="📊",
    layout="wide"
)

DATA_PATH = os.environ.get(
    "RETAIL_DATA_PATH",
    str(Path(__file__).resolve().parent.parent / "data" / "retail_cleaned.csv")
)

st.title("📊 RetailPulse")
st.caption("Retail analytics, forecasting and inventory intelligence")


@st.cache_data
def load_data(path):
    data = pd.read_csv(path)

    data["InvoiceDate"] = pd.to_datetime(
        data["InvoiceDate"], errors="coerce"
    )

    data = data.dropna(
        subset=["InvoiceDate", "StockCode", "Quantity"]
    ).copy()

    if "IsCancelled" in data.columns:
        data = data[data["IsCancelled"] == 0].copy()

    data["Date"] = data["InvoiceDate"].dt.floor("D")
    data["Revenue"] = (
        data["Revenue"]
        if "Revenue" in data.columns
        else data["Quantity"] * data["Price"]
        if "Price" in data.columns
        else 0
    )

    return data


if not os.path.exists(DATA_PATH):
    st.error(
        "Dataset not found. Update RETAIL_DATA_PATH "
        "to point to retail_cleaned.csv."
    )
    st.stop()

df = load_data(DATA_PATH)

page = st.sidebar.radio(
    "Navigation",
    [
        "Executive Overview",
        "Demand Forecasting",
        "Customer Analytics",
        "Inventory Optimization",
        "Monitoring & Alerts",
        "Reports & Export"
    ]
)

# Shared date filters
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()

date_range = st.sidebar.date_input(
    "Analysis date range",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date
)

if isinstance(date_range, (tuple, list)) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = df[
        (df["Date"].dt.date >= start_date) &
        (df["Date"].dt.date <= end_date)
    ].copy()
else:
    filtered = df.copy()

if filtered.empty:
    st.warning("No records found for this date range.")
    st.stop()

# Aggregate metrics
revenue = pd.to_numeric(
    filtered["Revenue"], errors="coerce"
).fillna(0).sum()

orders = filtered["Invoice"].nunique() if "Invoice" in filtered else 0

customers = (
    filtered["Customer ID"].nunique()
    if "Customer ID" in filtered else 0
)

daily_revenue = (
    filtered.groupby("Date", as_index=False)
    .agg(Revenue=("Revenue", "sum"))
    .sort_values("Date")
)

daily_demand = (
    filtered.groupby("Date", as_index=False)
    .agg(Demand=("Quantity", "sum"))
    .sort_values("Date")
)

# ---------------- Executive Overview ----------------
if page == "Executive Overview":
    st.header("Executive Overview")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Revenue", f"{revenue:,.2f}")
    c2.metric("Orders", f"{orders:,}")
    c3.metric("Customers", f"{customers:,}")
    c4.metric(
        "Units Sold",
        f"{filtered['Quantity'].clip(lower=0).sum():,.0f}"
    )

    fig = px.line(
        daily_revenue,
        x="Date", y="Revenue",
        title="Daily Revenue Trend"
    )
    st.plotly_chart(fig, use_container_width=True)

    top_products = (
        filtered.groupby("StockCode", as_index=False)
        .agg(Units=("Quantity", "sum"))
        .nlargest(10, "Units")
    )

    fig = px.bar(
        top_products,
        x="StockCode", y="Units",
        title="Top 10 Products by Units Sold"
    )
    st.plotly_chart(fig, use_container_width=True)

# ---------------- Demand Forecasting ----------------
elif page == "Demand Forecasting":
    st.header("Demand Forecasting")
    st.subheader("LSTM Revenue Forecast")

    forecast_days = st.slider(
        "Forecast horizon (days)",
        min_value=1,
        max_value=7,
        value=7
    )

    # Use the complete historical series, not only the
    # currently selected date-filtered data.
    history = (
        df.groupby("Date")["Revenue"]
        .sum()
        .sort_index()
        .asfreq("D", fill_value=0)
    )

    if len(history) < LOOKBACK:
        st.warning(
            f"At least {LOOKBACK} days of history are required."
        )
    else:
        predictions = forecast_revenue(
            history.iloc[-LOOKBACK:].values,
            horizon=forecast_days
        )

        forecast_dates = pd.date_range(
            start=history.index[-1] + pd.Timedelta(days=1),
            periods=forecast_days,
            freq="D"
        )

        forecast_df = pd.DataFrame({
            "Date": forecast_dates,
            "Forecast Revenue": predictions
        })

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=history.tail(60).index,
            y=history.tail(60).values,
            mode="lines",
            name="Historical Revenue"
        ))

        fig.add_trace(go.Scatter(
            x=forecast_df["Date"],
            y=forecast_df["Forecast Revenue"],
            mode="lines+markers",
            name="LSTM Forecast"
        ))

        fig.update_layout(
            title="Historical Revenue and LSTM Forecast",
            xaxis_title="Date",
            yaxis_title="Revenue",
            hovermode="x unified"
        )

        st.plotly_chart(fig, use_container_width=True)

        st.dataframe(
            forecast_df.round(2),
            use_container_width=True
        )

        st.download_button(
            "Download Forecast CSV",
            data=forecast_df.to_csv(index=False),
            file_name="retailpulse_lstm_forecast.csv",
            mime="text/csv"
        )


# ---------------- Customer Analytics ----------------
elif page == "Customer Analytics":
    st.header("Customer Analytics")

    if "Customer ID" not in filtered.columns:
        st.warning("Customer ID is unavailable in this dataset.")
    else:
        customer_data = filtered.dropna(
            subset=["Customer ID"]
        ).copy()

        rfm = customer_data.groupby("Customer ID").agg(
            Recency=("InvoiceDate", lambda x:
                     (customer_data["InvoiceDate"].max() - x.max()).days),
            Frequency=("Invoice", "nunique"),
            Monetary=("Revenue", "sum")
        ).reset_index()

        st.subheader("RFM customer overview")
        st.dataframe(
            rfm.sort_values("Monetary", ascending=False),
            use_container_width=True
        )

        fig = px.scatter(
            rfm,
            x="Frequency",
            y="Monetary",
            color="Recency",
            hover_data=["Customer ID"],
            title="Customer Frequency vs Monetary Value"
        )
        st.plotly_chart(fig, use_container_width=True)

        st.info(
            "RFM metrics are descriptive. Churn risk scores require "
            "a trained churn model and compatible customer features."
        )

# ---------------- Inventory Optimization ----------------
elif page == "Inventory Optimization":
    st.header("Inventory Optimization")

    inventory = (
        filtered.groupby("StockCode")
        .agg(
            TotalUnits=("Quantity", "sum"),
            ActiveDays=("Date", "nunique")
        )
        .reset_index()
    )

    inventory["AvgDailyDemand"] = (
        inventory["TotalUnits"] /
        inventory["ActiveDays"].clip(lower=1)
    )

    lead_time = st.number_input(
        "Lead time (days)", min_value=1, value=7
    )
    review_period = st.number_input(
        "Review period (days)", min_value=1, value=7
    )
    service_z = st.number_input(
        "Service-level Z-score", min_value=0.0,
        max_value=4.0, value=1.645
    )

    st.warning(
        "These are preliminary demand-based estimates. "
        "Actual stock on hand, incoming orders and reliable "
        "daily demand history are needed for actionable reorder advice."
    )

    # Approximate variability from product-level daily demand.
    daily_product = (
        filtered.groupby(["StockCode", "Date"])["Quantity"]
        .sum().reset_index()
    )

    variability = (
        daily_product.groupby("StockCode")["Quantity"]
        .std().fillna(0).rename("DailyStd")
    )

    inventory = inventory.merge(
        variability, on="StockCode", how="left"
    )

    inventory["SafetyStock"] = (
        service_z * inventory["DailyStd"] * np.sqrt(lead_time)
    )

    inventory["ReorderPoint"] = (
        inventory["AvgDailyDemand"] * lead_time
        + inventory["SafetyStock"]
    )

    inventory["TargetStock"] = (
        inventory["AvgDailyDemand"] *
        (lead_time + review_period)
        + inventory["SafetyStock"]
    )

    st.dataframe(
        inventory.sort_values(
            "ReorderPoint", ascending=False
        ),
        use_container_width=True
    )

    st.download_button(
        "Download inventory recommendations",
        inventory.to_csv(index=False).encode("utf-8"),
        file_name="inventory_recommendations.csv",
        mime="text/csv"
    )

# ---------------- Monitoring & Alerts ----------------
elif page == "Monitoring & Alerts":
    st.header("Monitoring & Alerts")

    st.metric("Rows in selected period", f"{len(filtered):,}")

    st.info(
        "This page is a monitoring placeholder until you connect "
        "your Evidently drift report, Airflow task status and "
        "saved model evaluation metrics."
    )

    drift_path = Path(__file__).resolve().parent.parent / "drift_outputs" / "retailpulse_data_drift_report.html"

    if os.path.exists(drift_path):
        st.success("A saved Evidently HTML report is available.")
        with open(drift_path, "rb") as f:
            st.download_button(
                "Download drift report",
                f.read(),
                file_name="retailpulse_data_drift_report.html",
                mime="text/html"
            )
    else:
        st.warning("No drift report found at the expected location.")

# ---------------- Reports & Export ----------------
elif page == "Reports & Export":
    st.header("Reports & Export")

    st.write("Export the filtered records or download a summary PDF.")

    st.download_button(
        "Download filtered data (CSV)",
        filtered.to_csv(index=False).encode("utf-8"),
        file_name="retailpulse_filtered_data.csv",
        mime="text/csv"
    )

    pdf_buffer = BytesIO()
    doc = SimpleDocTemplate(
        pdf_buffer, pagesize=A4
    )
    styles = getSampleStyleSheet()

    elements = [
        Paragraph("RetailPulse Analytics Summary", styles["Title"]),
        Spacer(1, 12),
        Paragraph(f"Revenue: {revenue:,.2f}", styles["Normal"]),
        Paragraph(f"Orders: {orders:,}", styles["Normal"]),
        Paragraph(f"Customers: {customers:,}", styles["Normal"]),
        Paragraph(
            f"Units sold: {filtered['Quantity'].clip(lower=0).sum():,.0f}",
            styles["Normal"]
        ),
        Spacer(1, 12)
    ]

    summary = [
        ["Metric", "Value"],
        ["Revenue", f"{revenue:,.2f}"],
        ["Orders", f"{orders:,}"],
        ["Customers", f"{customers:,}"]
    ]

    table = Table(summary)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("PADDING", (0, 0), (-1, -1), 6)
    ]))

    elements.append(table)
    doc.build(elements)

    st.download_button(
        "Download summary report (PDF)",
        pdf_buffer.getvalue(),
        file_name="retailpulse_summary.pdf",
        mime="application/pdf"
    )


Overwriting /content/app.py


In [4]:
import shutil

shutil.copy(
    "/content/app.py",
    os.path.join(APP_DIR, "app.py")
)

print("App saved to:", os.path.join(APP_DIR, "app.py"))

App saved to: /content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard/app.py


In [4]:
# Launch Streamlit

!streamlit run /content/app.py \
    --server.port 8502 \
    --server.headless true \
    > /content/streamlit.log 2>&1 &

In [6]:
%pip install -q streamlit plotly pandas numpy reportlab

In [7]:
import sys
import streamlit as st

print("Python executable:", sys.executable)
print("Streamlit version:", st.__version__)

Python executable: /usr/bin/python3
Streamlit version: 1.66.1


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import os

base_path = "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction"

print("Base folder exists:", os.path.exists(base_path))

if os.path.exists(base_path):
    print("Files and folders:")
    print(os.listdir(base_path))

Base folder exists: True
Files and folders:
['RetailPulse_Dashboard_Development.ipynb', 'online_retail.csv', 'daily_revenue.csv', '01_eda_timeseries.ipynb', '02_Prophet_model.ipynb', 'model', '03_LSTM.ipynb', 'retail_cleaned.csv', 'churn_model', '04_Churn_Prediction_XGBoost_SHAP.ipynb', 'inventory_recommendation.csv', 'product_demand_forecast.csv', 'top20_inventory_recommendations.csv', 'optuna_outputs', 'drift_outputs', 'airflow_retraining', '05_Feature_Importance_Optuna_Tuning.ipynb', '06_Inventory_Optimization_Forecasted_Demand.ipynb', '07_Drift_Detection_Evidently.ipynb', '08_Automated_Retraining_Airflow.ipynb', 'streamlit_dashboard']


In [21]:
dashboard_path = "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard"

print("Dashboard folder exists:", os.path.exists(dashboard_path))
print("Contents:", os.listdir(dashboard_path))

Dashboard folder exists: True
Contents: ['app.py', 'lstm_forecast.py', '__pycache__']


In [22]:
import os

APP_PATH = "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard/app.py"

print("File exists:", os.path.isfile(APP_PATH))
print("Full path:", APP_PATH)
print("File size:", os.path.getsize(APP_PATH), "bytes")

File exists: True
Full path: /content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard/app.py
File size: 11857 bytes


In [23]:
!python -m streamlit run "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard/app.py" --server.port 8502 --server.address 0.0.0.0



2026-10-26 14:26:08.558 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.225.42.216:8501

  Stopping...
  Stopping...


In [24]:
import os

APP_PATH = "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard/app.py"

print("App exists:", os.path.isfile(APP_PATH))

if os.path.isfile(APP_PATH):
    print("App size:", os.path.getsize(APP_PATH), "bytes")

App exists: True
App size: 11857 bytes


In [26]:
!python -m streamlit run "/content/drive/MyDrive/Colab Notebooks/Retail Sales Prediction/streamlit_dashboard/app.py" --server.port 8502 --server.address 0.0.0.0



2026-10-26 15:17:54.644 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://35.225.42.216:8502

  Stopping...


In [28]:
import requests

try:
    r = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Status code:", r.status_code)
    print("Response:", r.text)
except Exception as e:
    print(type(e).__name__, e)

ConnectionError HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: /_stcore/health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7d0cd4281090>: Failed to establish a new connection: [Errno 111] Connection refused'))
